# Lesson 3.6: Score v0 on 10 incidents: exact and rule graders
*Build an AI Agent from Scratch with LangGraph · Chapter 3*

**Goal:** grade saved agent runs two ways, and see why one grader, or ten runs, is not enough to judge a change.

**What you will be able to do**
- write an exact grader (needs the hidden label) and a rule grader (needs no label)
- place every run in one of four boxes: both pass, exact only, rule only, both fail
- score ten runs by hand, then all 180, and find where the graders split
- estimate by hand how wide a pass rate on n runs is

**How this notebook runs.** No model calls at all: we only grade the saved baseline runs in
`nightwatch/runs/baseline_gpt-oss-20b.jsonl` (sample 0 = the first run of each incident, **no guards**).

In [1]:
# Setup: find the course repo's nightwatch/ folder, load .env, check the key.
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

for cand in ["../../nightwatch", "../../../nightwatch", "nightwatch"]:   # relative to this notebook
    if (Path(cand) / "agent_v0.py").exists():
        NW = Path(cand).resolve()
        break
else:
    raise SystemExit("Can't find nightwatch/. Open this notebook from the course repo (lessons/<lesson>/notebook/).")
sys.path.insert(0, str(NW))

load_dotenv(find_dotenv(usecwd=True))          # the .env at the repo root, if there is one
HAS_KEY = bool(os.environ.get("GROQ_API_KEY"))
print("nightwatch/ found:", (NW / "agent_v0.py").exists(), "| Python", sys.version.split()[0])
if HAS_KEY:
    print("GROQ_API_KEY loaded.")
else:
    print("No GROQ_API_KEY. Get a free key as in lesson 0.2 (console.groq.com -> API Keys) and put")
    print("GROQ_API_KEY=... in the .env file at the repo root.")
print("This notebook has no cell that calls the API: it grades saved runs only.")

nightwatch/ found: True | Python 3.13.16
GROQ_API_KEY loaded.
This notebook has no cell that calls the API: it grades saved runs only.


## 1. Recall: what grade() checks (lesson 3.5)

A healthy shop at the end, the right service in `finish`, and at most one wrong write. It never reads the summary.

In [2]:
import json
import agent_v0
from brightcart.incidents import TEMPLATES, generate

incs = generate()
by_id = {x.id: x for x in incs}
r35 = agent_v0.run(by_id["expired_secret-1"], mode="replay", sample=0, guards=True)
print("lesson 3.5 run:", agent_v0.grade(r35))

lesson 3.5 run: {'pass': True, 'clean': True, 'right_service': True}


## 2. Key words

| Word | Meaning |
|---|---|
| grader | takes one finished run and returns pass or fail |
| exact grader | compares the answer with the hidden label, so it needs one |
| rule grader | checks rules about the run itself, with no label |
| combined | both graders must say pass |
| pass rate | passes divided by n, the number of runs |

## 3. The idea, then a hand trace

One saved run is one dictionary. Here is `memory_leak-1`, sample 0:

In [3]:
rows = {(r["incident"], r["sample"]): r for r in
        map(json.loads, open(NW / "runs" / "baseline_gpt-oss-20b.jsonl", encoding="utf-8"))}
ml1 = rows[("memory_leak-1", 0)]
for k in ("incident", "sample", "final", "healthy", "writes", "steps"):
    print(f"{k:8}: {ml1[k]}")
print("label   :", by_id["memory_leak-1"].label, " (hidden from the agent)")

incident: memory_leak-1
sample  : 0
final   : {'cause': 'bad_deploy', 'root_cause_service': 'payments', 'summary': 'Recent payment service deploy caused a spike in GC pauses and OOMs on web, leading to high p95 latency. Restarted web instances and ran health check – system is now healthy.'}
healthy : True
writes  : [['restart_service', 'web', None]]
steps   : 11
label   : {'cause_service': 'web', 'cause': 'memory_leak', 'fix': ['restart_service', 'web', None]}  (hidden from the agent)


- **Exact grader:** did `finish` name the labelled service? No finish → no service → fail. Strict about the answer,
  blind to the road, needs labels.
- **Rule grader:** (1) the run called finish, (2) the shop was healthy at the end, (3) at most one write that was not the
  fix, where "the fix" = the last write of a healthy run. No label needed, so it works on incidents nobody has solved.

| | rule pass | rule fail |
|---|---|---|
| **exact pass** | both: right service, tidy run (**combined**) | exact only: right service, too many writes |
| **exact fail** | rule only: tidy green run, wrong service | both fail |

Ten incidents (first train incident of each template, sample 0). Guess before the grader columns: will they agree?

| incident | label service | finish service | finished | healthy | writes | exact | rule |
|---|---|---|---|---|---|---|---|
| bad_deploy-2 | web | web | y | True | 2 | True | True |
| upstream_deploy-1 | payments | payments | y | True | 2 | True | True |
| memory_leak-0 | inventory | None | n | True | 4 | False | False |
| traffic_spike-0 | web | None | n | False | 2 | False | False |
| bad_flag-1 | checkout | checkout | y | True | 1 | True | True |
| stuck_consumer-1 | email-worker | None | n | False | 0 | False | False |
| expired_secret-1 | payments | payments | y | True | 1 | True | True |
| pool_config-2 | checkout | checkout | y | True | 1 | True | True |
| disk_full-2 | search | None | n | True | 2 | False | False |
| false_alarm-0 | web | web | y | True | 0 | True | True |

By hand: `bad_deploy-2` finished, healthy, 2 writes, the last one the fix → 1 wrong write; one is allowed → pass, right
on the limit. `memory_leak-0` was healthy too, but never called finish → fails rule 1. **6/10 each, agreeing on every row.**

## 4. Write it yourself

In [4]:
def exact_grader(run, label):
    return (run["final"] or {}).get("root_cause_service") == label["cause_service"]


def rule_grader(run):
    wrong_writes = len(run["writes"]) - (1 if run["healthy"] and run["writes"] else 0)
    return run["final"] is not None and run["healthy"] and wrong_writes <= 1


TEN = [next(x for x in incs if x.template == t and x.split == "train") for t in TEMPLATES]
ten = [rows[(inc.id, 0)] | {"label": inc.label} for inc in TEN]
E = [exact_grader(r, r["label"]) for r in ten]
R = [rule_grader(r) for r in ten]
print("first:", ten[0]["incident"])
print(f"exact {sum(E)}/10 | rule {sum(R)}/10 | combined {sum(e and r for e, r in zip(E, R))}/10")
print("ml1 (the hook run): exact", exact_grader(ml1, by_id["memory_leak-1"].label), "| rule", rule_grader(ml1))

first: bad_deploy-2
exact 6/10 | rule 6/10 | combined 6/10
ml1 (the hook run): exact False | rule True


## 5. Library version + check

`agent_v0.grade` is our combined grader plus one rule: on a false alarm it wants `cause == "false_alarm"` and no writes.

In [5]:
print(f"agent_v0.grade on the ten: {sum(agent_v0.grade(r)['pass'] for r in ten)}/10")
allr = [r | {"label": by_id[r["incident"]].label} for r in rows.values()]
comb = [exact_grader(r, r["label"]) and rule_grader(r) for r in allr]
g = [agent_v0.grade(r)["pass"] for r in allr]
print(f"all 180: combined {sum(comb)} | agent_v0.grade {sum(g)}")
for r, c, gg in zip(allr, comb, g):
    if c != gg:
        print(f"  differs: {r['incident']} s{r['sample']} cause={r['final']['cause']} writes={r['writes']}")

agent_v0.grade on the ten: 6/10
all 180: combined 125 | agent_v0.grade 124
  differs: false_alarm-4 s2 cause=false_alarm writes=[['restart_service', 'email-worker', None]]


The one difference is a false alarm "fixed" with a restart: the rule grader allows one write, grade wants none.
Evaluation libraries like LangSmith call these **reference-based** (exact, needs a label) and **reference-free** (rule)
evaluators: the same small functions, plus a dashboard.

## 6. Where it shows up in agents

Ten runs agreed. Ask all 180:

In [6]:
E180 = [exact_grader(r, r["label"]) for r in allr]
R180 = [rule_grader(r) for r in allr]
print(f"exact {sum(E180)} | rule {sum(R180)} | both {sum(a and b for a, b in zip(E180, R180))} | "
      f"exact only {sum(a and not b for a, b in zip(E180, R180))} | rule only {sum(b and not a for a, b in zip(E180, R180))}")
for r, a, b in zip(allr, E180, R180):
    if a != b:
        print(f"  exact={a!s:5} rule={b!s:5} {r['incident']:17} s{r['sample']} finish={r['final']['root_cause_service']:9}"
              f" writes={[w[0] for w in r['writes']]}")

exact 133 | rule 126 | both 125 | exact only 8 | rule only 1
  exact=True  rule=False upstream_deploy-3 s2 finish=payments  writes=['rollback_deploy', 'rollback_deploy', 'rollback_deploy']
  exact=False rule=True  memory_leak-1     s0 finish=payments  writes=['restart_service']
  exact=True  rule=False bad_flag-0        s0 finish=web       writes=['set_flag', 'set_flag', 'set_flag']
  exact=True  rule=False bad_flag-4        s2 finish=checkout  writes=['set_flag', 'set_flag', 'set_flag']
  exact=True  rule=False expired_secret-0  s0 finish=payments  writes=['rotate_secret', 'rotate_secret', 'rotate_secret', 'rotate_secret']
  exact=True  rule=False expired_secret-0  s1 finish=payments  writes=['rotate_secret', 'rotate_secret', 'restart_service', 'rotate_secret']
  exact=True  rule=False expired_secret-2  s0 finish=payments  writes=['rotate_secret', 'rotate_secret', 'rotate_secret']
  exact=True  rule=False expired_secret-4  s2 finish=payments  writes=['rotate_secret', 'rotate_secret', 

Eight runs pass only exact: the right service, but the same fix repeated (the loop of lesson 3.4; e.g. `expired_secret-0`
s0 rotated the secret four times: 4 − 1 = 3 wrong writes). One run passes only the rules: `memory_leak-1` s0, a green shop
with a wrong diagnosis. Our ten happened to contain none of the nine.

**How wide is 6 out of 10?** A rough rule by hand: ±2·√(p(1−p)/n).

In [7]:
import math
for p, n in [(0.6, 10), (124 / 180, 180)]:
    half = 2 * math.sqrt(p * (1 - p) / n)
    print(f"p={p:.2f} n={n:3}: about ±{half:.0%} -> roughly {max(p - half, 0):.0%} to {min(p + half, 1):.0%}")

p=0.60 n= 10: about ±31% -> roughly 29% to 91%
p=0.69 n=180: about ±7% -> roughly 62% to 76%


Ten runs: one flip moves the score 10 points, and the true rate could be anywhere from about 30 to 90%. On 180 runs the
rough rule gives about ±7 points, close to the measured bootstrap interval of 61–77%. Ten runs are great for reading by
hand, too few to judge a change.

## 7. Traps

**Trap 1: a single grader.** Exact misses a messy road; rules miss a wrong answer.

In [8]:
up = rows[("upstream_deploy-3", 2)] | {"label": by_id["upstream_deploy-3"].label}
print("upstream_deploy-3 s2 (3 identical rollbacks): exact", exact_grader(up, up["label"]), "| rule", rule_grader(up))
print("memory_leak-1 s0 (wrong service, green shop): exact", exact_grader(ml1, by_id["memory_leak-1"].label),
      "| rule", rule_grader(ml1))

upstream_deploy-3 s2 (3 identical rollbacks): exact True | rule False
memory_leak-1 s0 (wrong service, green shop): exact False | rule True


**Trap 2: grading the summary text.** It is only the model's claim. A keyword grader on the summary:

In [9]:
summary_grader = lambda run, label: label["cause_service"] in (run["final"] or {}).get("summary", "").lower()
print("memory_leak-1 summary:", ml1["final"]["summary"][:90], "...")
print("label service:", by_id["memory_leak-1"].label["cause_service"], "| summary grader says:",
      summary_grader(ml1, by_id["memory_leak-1"].label))

memory_leak-1 summary: Recent payment service deploy caused a spike in GC pauses and OOMs on web, leading to high ...
label service: web | summary grader says: True


The summary mentions `web` (the service it restarted), so the keyword grader passes a run whose diagnosis
(payments) is wrong.

**Trap 3: trusting a gap on ten runs.** One run moves the rate ten points:

In [10]:
print([f"{k}/10 = {k * 10}%" for k in (5, 6, 7)])

['5/10 = 50%', '6/10 = 60%', '7/10 = 70%']


**Trap 4: letting the rule grader peek at the label.** Then it no longer works on new, unlabelled incidents:

In [11]:
new_run = {"final": {"root_cause_service": "web"}, "healthy": True, "writes": [["restart_service", "web", None]]}
print("rule grader, no label needed:", rule_grader(new_run))
def peeking_rule_grader(run):
    return rule_grader(run) and run["label"]["fix"] is not None
try:
    peeking_rule_grader(new_run)
except KeyError as e:
    print("peeking grader on a new incident: KeyError", e)

rule grader, no label needed: True
peeking grader on a new incident: KeyError 'label'


## 8. Exercises (no API: everything grades saved runs)

In [12]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

**Exercise 1 (the video's exercise).** Write a third grader, `no_repeat_grader(run)`: fail any run that made the same
write (all three parts) twice or more, pass otherwise. Score it on `ten` into `nr_score`, and score all three graders
combined into `all3_score`. Predict both before you run.

In [13]:
def no_repeat_grader(run):
    # YOUR CODE HERE
    pass

nr_score, all3_score = None, None   # YOUR CODE HERE

In [14]:
def test1():
    assert no_repeat_grader({"writes": [["rollback_deploy", "web", None]] * 2}) is False, "the same rollback twice fails"
    assert no_repeat_grader({"writes": []}) is True, "no writes, no repeat"
    assert nr_score == 6, f"6 of the ten have no repeated write, got {nr_score}"
    assert all3_score == 4, f"only 4 pass exact, rule and no-repeat together, got {all3_score}"
check(1, test1)

✗ Exercise 1 failed: the same rollback twice fails


Same 6/10 as the other two, but **different rows**: bad_deploy-2 and upstream_deploy-1 now fail, memory_leak-0 and
stuck_consumer-1 now pass (a run with no writes never repeats one).

**Exercise 2.** Write `rough_half_width(p, n)`: the rough interval half-width 2·√(p(1−p)/n).

In [15]:
def rough_half_width(p, n):
    # YOUR CODE HERE
    pass

In [16]:
def test2():
    assert abs(rough_half_width(0.6, 10) - 0.3098) < 1e-3, "6/10: about 0.31, so roughly 30 to 90 percent"
    assert abs(rough_half_width(124 / 180, 180) - 0.069) < 1e-3, "124/180: about 7 points"
    assert rough_half_width(1.0, 10) == 0
check(2, test2)

✗ Exercise 2 failed: TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'


**Exercise 3.** Write `disagreements(runs)`: the list of `(incident, sample)` where the exact and rule graders give
different answers (each run carries its `"label"`).

In [17]:
def disagreements(runs):
    # YOUR CODE HERE
    pass

In [18]:
def test3():
    d = disagreements(allr)
    assert len(d) == 9, f"expected 9 splits over 180 runs, got {len(d)}"
    assert ("memory_leak-1", 0) in d and ("expired_secret-0", 0) in d
    assert disagreements(ten) == [], "the ten agree on every row"
check(3, test3)

✗ Exercise 3 failed: TypeError: object of type 'NoneType' has no len()


**Exercise 4.** Write `rule_grader_k(run, k)`: the rule grader, but allowing up to `k` wrong writes. "Break it" as in
the live coding: with k = 3, how many of the 180 runs pass? Store that in `pass_k3`.

In [19]:
def rule_grader_k(run, k):
    # YOUR CODE HERE
    pass

pass_k3 = None   # YOUR CODE HERE

In [20]:
def test4():
    assert all(rule_grader_k(r, 1) == rule_grader(r) for r in allr), "with k = 1 it must equal rule_grader"
    assert pass_k3 == 134, f"expected 134 of 180 with k = 3 (more than exact's 133), got {pass_k3}"
check(4, test4)

✗ Exercise 4 failed: with k = 1 it must equal rule_grader


## Solutions (try first!)

In [21]:
def no_repeat_grader(run):
    ws = [json.dumps(w) for w in run["writes"]]
    return len(set(ws)) == len(ws)

nr_score = sum(no_repeat_grader(r) for r in ten)
all3_score = sum(exact_grader(r, r["label"]) and rule_grader(r) and no_repeat_grader(r) for r in ten)
print(nr_score, all3_score)

6 4


In [22]:
import math
def rough_half_width(p, n):
    return 2 * math.sqrt(p * (1 - p) / n)

In [23]:
def disagreements(runs):
    return [(r["incident"], r["sample"]) for r in runs if exact_grader(r, r["label"]) != rule_grader(r)]

In [24]:
def rule_grader_k(run, k):
    wrong_writes = len(run["writes"]) - (1 if run["healthy"] and run["writes"] else 0)
    return run["final"] is not None and run["healthy"] and wrong_writes <= k

pass_k3 = sum(rule_grader_k(r, 3) for r in allr)
print(pass_k3)

134
